# Coupling MPI Applications with PyTorch Inference and Training

**Estimated time:** ~45-60 minutes  
**Format:** 5 exercises. Each includes demo code, a small coding task, and a hidden solution.

---

## Session goals

This notebook builds a prototype coupled AI+HPC workflow using DragonHPC.
You will combine MPI-style application orchestration with PyTorch-based
inference and training.

You will practice how to:

- launch MPI applications with Dragon ProcessGroup
- capture selected rank output for downstream AI stages
- transform HPC outputs into model-ready tensors
- run PyTorch inference and training in cooperating processes
- compose an end-to-end coupled workflow

The exercises are written for beginners and intermediate users, while
providing extensible patterns for advanced users.

---

## Setup - run this first

Start Jupyter from a Dragon-enabled environment (for example with
`dragon-jupyter`) and run the next cell.

In [ ]:
import os
import re
import queue
import socket
from pathlib import Path

import dragon
import multiprocessing as mp
import dragon.native as dn
import torch

from dragon.infrastructure.facts import PMIBackend
from dragon.native.process import Process, ProcessTemplate, MSG_PIPE, MSG_DEVNULL
from dragon.native.process_group import ProcessGroup

try:
    mp.set_start_method("dragon")
except RuntimeError:
    pass

print("Dragon + PyTorch setup complete")
print("PyTorch version:", torch.__version__)

mpi_hello = Path("../dragon_native/mpi/mpi_hello").resolve()
print("mpi_hello exists:", mpi_hello.exists())

If `mpi_hello` is missing, build it once:

```bash
cd ../dragon_native/mpi
make
```

Then rerun Cell 2.

---

## Exercise 1 - Launch MPI and capture head-rank output

**Background:**

A common coupling pattern is to run an MPI application and capture only rank 0
stdout via `MSG_PIPE` while suppressing other ranks with `MSG_DEVNULL`.

Demo pattern:

```python
pg = ProcessGroup(restart=False, pmi=PMIBackend.CRAY)
pg.add_process(nproc=1, template=ProcessTemplate(target=exe, stdout=MSG_PIPE))
pg.add_process(nproc=n-1, template=ProcessTemplate(target=exe, stdout=MSG_DEVNULL))
```

**Your task:**

1. Write `run_mpi_capture(exe_path, total_ranks)`
2. Launch MPI with rank 0 piped and others devnull
3. Return a list of captured lines from rank 0
4. Ensure group is joined and closed

In [ ]:
# -- Exercise 1 -- your code here -----------------------------------------------

def run_mpi_capture(exe_path, total_ranks):
    pass

<details>
<summary><b>▶ Show Solution</b></summary>

```python
def run_mpi_capture(exe_path, total_ranks):
    pg = ProcessGroup(restart=False, pmi=PMIBackend.CRAY)
    pg.add_process(
        nproc=1,
        template=ProcessTemplate(target=exe_path, args=(), cwd=str(Path(exe_path).parent), stdout=MSG_PIPE),
    )
    if total_ranks > 1:
        pg.add_process(
            nproc=total_ranks - 1,
            template=ProcessTemplate(target=exe_path, args=(), cwd=str(Path(exe_path).parent), stdout=MSG_DEVNULL),
        )

    pg.init()
    pg.start()

    lines = []
    for puid in pg.puids:
        child = Process(None, ident=puid)
        if child.stdout_conn:
            try:
                while True:
                    lines.append(child.stdout_conn.recv().strip())
            except EOFError:
                pass

    pg.join()
    pg.close()
    return lines

exe = str(Path("../dragon_native/mpi/mpi_hello").resolve())
if Path(exe).exists():
    out = run_mpi_capture(exe, total_ranks=4)
    print("captured lines:", len(out))
    print(out[0] if out else "<no output>")
else:
    print("Build mpi_hello first in ../dragon_native/mpi")
```

</details>

---

## Exercise 2 - Convert HPC output into PyTorch features

**Background:**

AI coupling usually requires converting application output text into numeric
features. Here we build a tiny featurizer from captured lines.

Demo pattern:

```python
def line_to_feature(line):
    return [len(line), line.lower().count("rank")]
```

**Your task:**

1. Write `build_feature_tensor(lines)`
2. Produce tensor shape `[N, 3]` with features:
   `[line_length, rank_count, digit_count]`
3. Return `torch.float32` tensor
4. Test with 3 sample lines

In [ ]:
# -- Exercise 2 -- your code here -----------------------------------------------

def build_feature_tensor(lines):
    pass

<details>
<summary><b>▶ Show Solution</b></summary>

```python
def build_feature_tensor(lines):
    rows = []
    for line in lines:
        line_length = len(line)
        rank_count = line.lower().count("rank")
        digit_count = sum(ch.isdigit() for ch in line)
        rows.append([line_length, rank_count, digit_count])
    if not rows:
        return torch.zeros((0, 3), dtype=torch.float32)
    return torch.tensor(rows, dtype=torch.float32)

sample = [
    "Hello, world, rank 0 on host x",
    "rank 1 finished step 12",
    "no rank keyword here",
]
x = build_feature_tensor(sample)
print(x)
print("shape:", tuple(x.shape))
```

</details>

---

## Exercise 3 - PyTorch inference service process

**Background:**

A lightweight inference process can consume feature tensors and return
predictions through a queue. This mirrors online scoring in coupled workflows.

Demo pattern:

```python
model = torch.nn.Sequential(torch.nn.Linear(3, 8), torch.nn.ReLU(), torch.nn.Linear(8, 1))
y = model(x)
```

**Your task:**

1. Write `inference_worker(in_q, out_q)`
2. Worker should read feature tensors from `in_q` until it receives `None`
3. Run model forward pass with `torch.no_grad()`
4. Put predictions on `out_q`
5. Launch worker, send one batch, print predictions

In [ ]:
# -- Exercise 3 -- your code here -----------------------------------------------

def inference_worker(in_q, out_q):
    pass

<details>
<summary><b>▶ Show Solution</b></summary>

```python
def inference_worker(in_q, out_q):
    model = torch.nn.Sequential(
        torch.nn.Linear(3, 8),
        torch.nn.ReLU(),
        torch.nn.Linear(8, 1),
    )
    model.eval()

    while True:
        batch = in_q.get()
        if batch is None:
            break
        with torch.no_grad():
            pred = model(batch)
        out_q.put(pred)

in_q = mp.Queue()
out_q = mp.Queue()
worker = dn.Process(target=inference_worker, args=(in_q, out_q))
worker.start()

x = torch.tensor([[10.0, 1.0, 2.0], [20.0, 0.0, 4.0]], dtype=torch.float32)
in_q.put(x)
print(out_q.get())

in_q.put(None)
worker.join()
```

</details>

---

## Exercise 4 - PyTorch training on streamed HPC batches

**Background:**

Training can run as a separate stage receiving mini-batches generated from
HPC output. Here we train a tiny regression model with MSE loss.

Demo pattern:

```python
opt = torch.optim.Adam(model.parameters(), lr=1e-2)
loss = torch.nn.functional.mse_loss(pred, target)
```

**Your task:**

1. Write `trainer_worker(train_q, result_q)`
2. Repeatedly read `(x_batch, y_batch)` until sentinel `None`
3. Perform one optimizer step per batch
4. Send final scalar loss to `result_q`
5. Demonstrate with 5 synthetic batches

In [ ]:
# -- Exercise 4 -- your code here -----------------------------------------------

def trainer_worker(train_q, result_q):
    pass

<details>
<summary><b>▶ Show Solution</b></summary>

```python
def trainer_worker(train_q, result_q):
    model = torch.nn.Sequential(
        torch.nn.Linear(3, 16),
        torch.nn.ReLU(),
        torch.nn.Linear(16, 1),
    )
    opt = torch.optim.Adam(model.parameters(), lr=1e-2)

    final_loss = None
    while True:
        item = train_q.get()
        if item is None:
            break
        x_batch, y_batch = item
        pred = model(x_batch)
        loss = torch.nn.functional.mse_loss(pred, y_batch)
        opt.zero_grad()
        loss.backward()
        opt.step()
        final_loss = float(loss.item())

    result_q.put(final_loss if final_loss is not None else float("nan"))

train_q = mp.Queue()
result_q = mp.Queue()
trainer = dn.Process(target=trainer_worker, args=(train_q, result_q))
trainer.start()

for i in range(5):
    x = torch.randn(8, 3)
    y = (0.3 * x[:, :1] + 0.1 * x[:, 1:2] - 0.2 * x[:, 2:3])
    train_q.put((x, y))

train_q.put(None)
trainer.join()
print("final loss:", result_q.get())
```

</details>

---

## Exercise 5 - Build a prototype coupled AI+HPC workflow

**Background:**

This capstone exercise couples an MPI producer with PyTorch training and
inference workers in one mini workflow. The goal is clean stage boundaries
and robust process orchestration.

Demo architecture:

```text
MPI ProcessGroup -> captured lines -> feature builder -> train worker + infer worker
```

**Your task:**

1. Implement `run_coupled_workflow(exe_path)`
2. Capture MPI rank-0 lines using `run_mpi_capture`
3. Build feature tensor with `build_feature_tensor`
4. Send one batch to trainer and one batch to inference worker
5. Print workflow summary: number of lines, final training loss, prediction shape

In [ ]:
# -- Exercise 5 -- your code here -----------------------------------------------

def run_coupled_workflow(exe_path):
    pass

<details>
<summary><b>▶ Show Solution</b></summary>

```python
def run_coupled_workflow(exe_path):
    lines = run_mpi_capture(exe_path, total_ranks=4)
    x = build_feature_tensor(lines)

    if x.shape[0] == 0:
        print("No MPI lines captured; aborting workflow")
        return

    y = (0.05 * x[:, :1] + 0.01 * x[:, 1:2] + 0.02 * x[:, 2:3])

    train_q = mp.Queue()
    train_result_q = mp.Queue()
    infer_in_q = mp.Queue()
    infer_out_q = mp.Queue()

    trainer = dn.Process(target=trainer_worker, args=(train_q, train_result_q))
    inferer = dn.Process(target=inference_worker, args=(infer_in_q, infer_out_q))

    trainer.start()
    inferer.start()

    train_q.put((x, y))
    train_q.put(None)

    infer_in_q.put(x)
    pred = infer_out_q.get()
    infer_in_q.put(None)

    trainer.join()
    inferer.join()

    final_loss = train_result_q.get()
    print("Workflow summary")
    print("  captured lines:", len(lines))
    print("  final training loss:", final_loss)
    print("  prediction shape:", tuple(pred.shape))

exe = str(Path("../dragon_native/mpi/mpi_hello").resolve())
if Path(exe).exists():
    run_coupled_workflow(exe)
else:
    print("Build mpi_hello first in ../dragon_native/mpi")
```

</details>

---

## Summary

You built a prototype coupled AI+HPC pipeline using DragonHPC ProcessGroup
and PyTorch stages for both inference and training.

| Concept | API |
|---|---|
| MPI launch orchestration | `ProcessGroup(..., pmi=PMIBackend.CRAY)` |
| Rank output capture | `MSG_PIPE` + `stdout_conn.recv()` |
| Quiet non-head ranks | `MSG_DEVNULL` |
| Feature conversion | Python parsing -> `torch.tensor` |
| Inference service | separate process + queues + `torch.no_grad()` |
| Training stage | separate process + optimizer + loss backward |
| Workflow composition | MPI producer + AI workers + parent coordinator |

### Next steps

- Replace toy features with domain-specific simulation outputs.
- Swap toy models for your production PyTorch model checkpoints.
- Add multi-node placement policies and telemetry for throughput tuning.